# Lesson 4: How to Bring Your Transformers Model to vLLM

In Lesson 3, we saw that the Transformers backend can load any *compatible* Hugging Face model in vLLM. But what makes a model compatible?

In this lesson, we'll reverse-engineer **OLMoE** to understand exactly what's needed. We'll walk through each requirement in the actual source code, and contrast with a model that *isn't* compatible.

By the end, you'll have a clear checklist for making any Transformers model work with vLLM.

## The Compatibility Checklist

A Transformers model needs these things to work with vLLM's Transformers backend:

1. `_supports_attention_backend = True` on the `PreTrainedModel`
2. `ALL_ATTENTION_FUNCTIONS` dispatch in the attention forward method
3. `**kwargs` pass-through at every level of the forward chain
4. **(MoE models only)** A replaceable `experts` submodule with a separate router, so `MoEMixin` can swap it for vLLM's `FusedMoE`
5. `base_model_tp_plan` in the config (tensor parallelism)
6. `base_model_pp_plan` in the config (pipeline parallelism)

Let's examine each one in OLMoE's code.

## Step 1: The Gate — `_supports_attention_backend`

This class attribute is the first check vLLM performs. If it's `False` (or missing), the model is invisible to the Transformers backend.

vLLM checks this via the `is_backend_compatible()` method when resolving which model class to use.

In [ ]:
from transformers import OlmoeForCausalLM

# OLMoE supports the attention backend
print(f"OlmoeForCausalLM._supports_attention_backend = {OlmoeForCausalLM._supports_attention_backend}")

# Contrast: DiffLlama does NOT support it
from transformers import DiffLlamaForCausalLM
print(f"DiffLlamaForCausalLM._supports_attention_backend = {DiffLlamaForCausalLM._supports_attention_backend}")

Why doesn't DiffLlama support the backend? Because its attention mechanism is fundamentally different — it computes **differential attention** with two separate attention operations combined via learned lambda parameters. This custom pattern doesn't fit the standard `ALL_ATTENTION_FUNCTIONS` dispatch.

Let's see where OLMoE sets this flag:

In [ ]:
import inspect
from transformers.models.olmoe.modular_olmoe import OlmoePreTrainedModel

source = inspect.getsource(OlmoePreTrainedModel)
print(source)

## Step 2: The Attention Interface — `ALL_ATTENTION_FUNCTIONS`

This is the core mechanism that bridges Transformers and vLLM. Instead of hardcoding the attention computation, the model must call:

```python
attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface(
    self.config._attn_implementation, eager_attention_forward
)
attn_output, attn_weights = attention_interface(
    self, query, key, value, attention_mask, **kwargs
)
```

`ALL_ATTENTION_FUNCTIONS` behaves like a dict, so the equivalent subscript form also works:

```python
attention_interface = ALL_ATTENTION_FUNCTIONS[self.config._attn_implementation]
```

The `get_interface(...)` helper is just a thin wrapper that adds extra validation (checking the key exists, falling back to a default like `eager_attention_forward`, etc.) — the actual lookup is a plain mapping access.

This allows different backends (SDPA, Flash Attention, **vLLM**) to be swapped in transparently. The `**kwargs` at the end is critical — that's how `attention_instances` reaches vLLM's attention function.

Let's look at OLMoE's attention forward:

In [ ]:
from transformers.models.olmoe.modular_olmoe import OlmoeAttention

source = inspect.getsource(OlmoeAttention.forward)
print(source)

Notice the key elements:
1. The signature includes `**kwargs: Unpack[TransformersKwargs]`
2. It calls `ALL_ATTENTION_FUNCTIONS.get_interface(self.config._attn_implementation, ...)` to get the attention function
3. It passes `**kwargs` to the attention interface

Now contrast with DiffLlama, which does **not** use this pattern:

In [ ]:
from transformers.models.diffllama.modular_diffllama import DiffLlamaAttention

source = inspect.getsource(DiffLlamaAttention.forward)
print(source)

DiffLlama computes attention weights directly with `torch.matmul` and `softmax` instead of dispatching through `ALL_ATTENTION_FUNCTIONS`. This is because differential attention needs two separate attention computations combined with learned lambda parameters — it can't be reduced to a single standard attention call.

## Step 3: The `**kwargs` Chain

vLLM's `Base.forward()` passes `attention_instances` as a keyword argument:

```python
outputs = self.model(
    input_ids=input_ids,
    ...
    attention_instances=self.attention_instances,  # <-- This!
    ...
)
```

Note that vLLM only instantiates the base **Model** class (via `AutoModel.from_config()`) — the `ForCausalLM` class is never used. The language-model head is added separately by vLLM's `CausalMixin`. So the `**kwargs` chain that matters starts at `Model.forward`, not `ForCausalLM.forward`:

```
Model.forward(**kwargs)
  -> DecoderLayer.forward(**kwargs)
    -> Attention.forward(**kwargs)
      -> ALL_ATTENTION_FUNCTIONS["vllm"](**kwargs)
```

If **any** level in this chain doesn't accept and forward `**kwargs`, the attention instances are lost and inference fails.

Let's verify that OLMoE passes `**kwargs` at every level:

In [ ]:
from transformers.models.olmoe.modular_olmoe import (
    OlmoeModel,
    OlmoeDecoderLayer,
    OlmoeAttention,
)

classes = [
    ("OlmoeModel", OlmoeModel),
    ("OlmoeDecoderLayer", OlmoeDecoderLayer),
    ("OlmoeAttention", OlmoeAttention),
]

for name, cls in classes:
    sig = inspect.signature(cls.forward)
    params = list(sig.parameters.keys())
    has_kwargs = any(
        p.kind == inspect.Parameter.VAR_KEYWORD
        for p in sig.parameters.values()
    )
    marker = "has **kwargs" if has_kwargs else "MISSING **kwargs!"
    print(f"{name}.forward({', '.join(params)}) -> {marker}")

All three levels forward `**kwargs`, so `attention_instances` has an unbroken path from `Base.forward` down to the attention function.

## Step 4 (MoE only): Replaceable Experts

For MoE models, vLLM's backend swaps the Transformers `experts` submodule for vLLM's `FusedMoE`. Since Transformers v5 experts aren't stored per-expert anymore — they're already packed 3D tensors driven by a grouped-MM kernel — so the motivation *isn't* about fusing N tiny expert MLPs. The reason is that `FusedMoE` plugs the MoE layer into vLLM's serving features: **expert parallelism (EP)**, **expert-parallel load balancing (EPLB)**, and vLLM's quantization and parallel-init machinery. Stay on the Transformers expert module and those features aren't available on this layer.

From Lesson 3, recall that this swap happens inside `MoEMixin.recursive_replace()`, which runs during `Base.__init__` *before* the tensor-parallel linear replacement. That ordering matters: once the experts are gone, the TP plan entries for the expert weights become irrelevant on the vLLM path — `FusedMoE` is already there and handles its own sharding.

For the swap to work, the model's MoE block has to expose its experts in a shape the mixin can recognize. Let's look at OLMoE's MoE block:

In [ ]:
from transformers.models.olmoe.modeling_olmoe import OlmoeExperts, OlmoeSparseMoeBlock

print(inspect.getsource(OlmoeSparseMoeBlock))
print(f"class OlmoeExperts(nn.Module):\n{inspect.getsource(OlmoeExperts.__init__)}")

Three properties make this MoE block replaceable:

1. **The experts live in a submodule named `experts`.** `MoEMixin.recursive_replace()` walks the model tree looking specifically for modules with that attribute name — if it were called `moe_experts` or `mlp`, the backend would skip it.
2. **The router is a *sibling* of the experts, not bundled inside them.** `self.gate` runs first and produces `(top_k_weights, top_k_index)`; those are then passed into `self.experts(...)`. Because the backend only replaces the `experts` submodule (not the whole block), it can intercept the routing decisions Transformers has already made via a `custom_routing_function` instead of recomputing them inside `FusedMoE`.
3. **Expert weights are packed 3D tensors with recognizable names.** `OlmoeExperts` inherits `MixtralExperts`, which stores weights as `gate_up_proj` of shape `(num_experts, 2 * intermediate_dim, hidden_dim)` and `down_proj` of shape `(num_experts, hidden_dim, intermediate_dim)`. vLLM's `FusedMoE` knows how to load weights from these exact names and shapes.

If any one of these broke — experts named differently, router fused into the expert module, or weights stored as a `nn.ModuleList` of per-expert linears that the mixin doesn't recognize — the MoE block wouldn't be replaceable and you'd either get much slower expert computation or outright failure.

## Step 5: Tensor Parallel Plan

The `base_model_tp_plan` in the config tells vLLM (and PyTorch's tensor parallelism) how to shard each linear layer across GPUs:

- **`"colwise"`** — column-parallel: each GPU gets a slice of the output dimension. Used for Q, K, V, gate, and up projections.
- **`"rowwise"`** — row-parallel: each GPU gets a slice of the input dimension. Used for O and down projections.
- **`"packed_colwise"`** — column-parallel for fused/packed weight matrices.

OLMoE has some special entries due to its Q/K norm:

In [ ]:
from transformers import OlmoeConfig

print("OlmoeConfig.base_model_tp_plan:")
for pattern, style in OlmoeConfig.base_model_tp_plan.items():
    print(f"  {pattern:50s} -> {style}")

Notice the `"colwise_gather_output"` for Q, K, V projections and `"rowwise_split_input"` for O projections. These variants are needed because OLMoE applies RMSNorm to Q and K *after* projection — the norm needs the full (gathered) output, so the standard column-parallel approach (which keeps outputs sharded) doesn't work directly.

You'll also see MoE expert entries in the plan (`"moe_tp_experts"` for the expert module, `"packed_colwise"` / `"rowwise"` for the individual expert weights) — but the Transformers backend **doesn't actually use those** for sharding. `MoEMixin.recursive_replace()` has already swapped the `experts` module for vLLM's `FusedMoE` by the time TP is applied, and `FusedMoE` handles its own tensor-parallel sharding internally. The plan entries are still useful when the native Transformers model is run with PyTorch's TP (e.g. for training), just not on the vLLM path.

## Step 6: Pipeline Parallel Plan

The `base_model_pp_plan` defines how to split the model across pipeline stages. It maps module names to their position in the pipeline. Unlike TP (which splits layers), PP splits the model into sequential stages.

OLMoE inherits its PP plan from its parent class. Let's check:

In [ ]:
print("OlmoeConfig.base_model_pp_plan:")
for module, (inputs, outputs) in OlmoeConfig.base_model_pp_plan.items():
    print(f"  {module:20s} inputs={inputs}, outputs={outputs}")

## Putting It All Together: Before vs After

Here's what a model looks like **without** backend support vs **with** it.

**Before — not compatible with the vLLM backend:**

```python
class MyConfig(PreTrainedConfig):
    # No TP plan
    # No PP plan
    ...

class MyAttention(nn.Module):
    def forward(self, hidden_states, attention_mask=None):  # No **kwargs!
        q = self.q_proj(hidden_states)
        k = self.k_proj(hidden_states)
        v = self.v_proj(hidden_states)
        # Hardcoded attention computation
        attn = torch.matmul(q, k.T) / math.sqrt(d)
        attn = softmax(attn)
        output = torch.matmul(attn, v)
        return self.o_proj(output)

class MyMoEBlock(nn.Module):                             # MoE block shape that
    def __init__(self, config):                          # the backend can't replace
        super().__init__()
        # Router fused inside the expert module, and experts named `moe_experts`
        # (not `experts`). MoEMixin will skip this block.
        self.moe_experts = FusedRouterAndExperts(config)

    def forward(self, hidden_states):
        return self.moe_experts(hidden_states)

class MyPreTrainedModel(PreTrainedModel):
    # Missing: _supports_attention_backend = True
    ...
```

**After — compatible with the vLLM backend:**

```python
class MyConfig(PreTrainedConfig):
    base_model_tp_plan = {                              # + TP plan
        "layers.*.self_attn.q_proj": "colwise",
        "layers.*.self_attn.k_proj": "colwise",
        "layers.*.self_attn.v_proj": "colwise",
        "layers.*.self_attn.o_proj": "rowwise",
        "layers.*.mlp.gate_proj":    "colwise",
        "layers.*.mlp.up_proj":      "colwise",
        "layers.*.mlp.down_proj":    "rowwise",
    }

class MyAttention(nn.Module):
    def forward(self, hidden_states, attention_mask=None,
                **kwargs):                              # + **kwargs
        q = self.q_proj(hidden_states)
        k = self.k_proj(hidden_states)
        v = self.v_proj(hidden_states)
        # Dispatch through ALL_ATTENTION_FUNCTIONS      # + dispatch
        attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )
        output, weights = attention_interface(
            self, q, k, v, attention_mask, **kwargs     # + **kwargs
        )
        return self.o_proj(output)

class MyExperts(nn.Module):                             # + packed 3D expert weights
    def __init__(self, config):
        super().__init__()
        self.gate_up_proj = nn.Parameter(torch.empty(
            config.num_experts, 2 * config.intermediate_dim, config.hidden_dim
        ))
        self.down_proj = nn.Parameter(torch.empty(
            config.num_experts, config.hidden_dim, config.intermediate_dim
        ))

    def forward(self, hidden_states, top_k_index, top_k_weights):
        ...                                             # grouped-MM over packed weights

class MyMoEBlock(nn.Module):                            # + replaceable MoE block
    def __init__(self, config):
        super().__init__()
        self.gate = MyRouter(config)                    # router is a SIBLING
        self.experts = MyExperts(config)                # submodule named `experts`

    def forward(self, hidden_states):
        _, top_k_weights, top_k_index = self.gate(hidden_states)
        return self.experts(hidden_states, top_k_index, top_k_weights)

class MyPreTrainedModel(PreTrainedModel):
    _supports_attention_backend = True                  # + flag
    ...
```

## Verifying It Works

Let's verify the full path: from model registry resolution to actual generation.

In [ ]:
from vllm.model_executor.models.registry import _VLLM_MODELS

# Show how vLLM resolves OLMoE's architecture
from transformers import AutoConfig

config = AutoConfig.from_pretrained("allenai/OLMoE-1B-7B-0125-Instruct")
arch = config.architectures[0]
print(f"Model architectures: {config.architectures}")
print(f"  -> '{arch}' is in the native registry: {arch in _VLLM_MODELS}")
print(f"  -> So vLLM uses the native implementation by default")
print(f"  -> With model_impl='transformers', it uses TransformersMoEForCausalLM instead")

In [ ]:
# End-to-end: generate with the Transformers backend
from vllm import LLM, SamplingParams

MODEL_NAME = "allenai/OLMoE-1B-7B-0125-Instruct"
# MODEL_NAME = "Qwen/Qwen3-0.6B"  # Backup if there isn't enough memory for OLMoE

llm = LLM(
    model=MODEL_NAME,
    model_impl="transformers",
)

outputs = llm.generate(
    ["What makes mixture-of-experts models efficient?"],
    SamplingParams(max_tokens=100, temperature=0.7),
)

print(outputs[0].outputs[0].text)

## Common Pitfalls

When making a model compatible with the Transformers backend, watch out for:

1. **Missing `**kwargs` at any level** — The most common issue. If `OlmoeModel.forward` accepted `**kwargs` but `OlmoeDecoderLayer.forward` didn't, `attention_instances` would be silently dropped.

2. **Custom attention not using `ALL_ATTENTION_FUNCTIONS`** — Models that compute attention inline (like DiffLlama) can't be dispatched to vLLM's kernels. The model must use the standard dispatch pattern.

3. **Incorrect TP plans** — Misspecifying `"colwise"` vs `"rowwise"` will produce wrong results silently. Remember: projections that *increase* dimension (Q, K, V, gate, up) are typically `"colwise"`, and projections that *decrease* dimension (O, down) are `"rowwise"`.

4. **Non-standard attention mask handling** — Models that manipulate attention weights directly (e.g., adding positional bias to attention scores after softmax) may not be expressible through the standard attention interface.

## Summary

To make a Transformers model work with vLLM's Transformers backend:

| # | Requirement | Where | Why |
|---|-------------|-------|-----|
| 1 | `_supports_attention_backend = True` | `PreTrainedModel` class | Gate check for backend compatibility |
| 2 | `ALL_ATTENTION_FUNCTIONS` dispatch | Attention `.forward()` | Allows vLLM to inject its attention kernels |
| 3 | `**kwargs` pass-through | Every `.forward()` in the chain | Routes `attention_instances` to the attention function |
| 4 | Replaceable `experts` submodule with sibling router (MoE only) | MoE block | Lets `MoEMixin` swap the experts for vLLM's `FusedMoE` |
| 5 | `base_model_tp_plan` | Config class | Tells vLLM how to shard layers for tensor parallelism |
| 6 | `base_model_pp_plan` | Config class | Tells vLLM how to split the model for pipeline parallelism |

Contributing these changes upstream to Transformers benefits the entire ecosystem — once a model is compatible, it works in vLLM, SGLang, and any other engine that uses the Transformers backend.